# 9 — Same-slide integration

**Only** for the case where both modalities imaged the *same physical section*. Then the
cells are literally the same cells and a genuine paired protein+RNA matrix is recoverable —
everything the sequential mode has to approximate becomes a measurement instead.

Set `[integration] mode = same_slide` in `config.ini`. The guard is a hard error in both
directions: `sameslide` refuses to run in sequential mode, and the sequential
approximations refuse to run here.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from phenocycler import load_config

cfg = load_config(pathlib.Path.cwd().parent / 'config.ini', integration_mode='same_slide')
print('mode:', cfg.integration_mode)

## The guard

Worth seeing it fire once. Cell-to-cell pairing across serial sections is not a
measurement, however good the registration.

In [ ]:
from phenocycler.integration.sameslide import require_same_slide, ModeError

seq = load_config(pathlib.Path.cwd().parent / 'config.ini', integration_mode='sequential')
try:
    require_same_slide(seq)
except ModeError as exc:
    print('correctly refused:\n')
    print(exc)

## Prep, structures and registration

Same as sequential up to this point — but registration here must be accurate to a few
microns rather than a few tens, because cells are matched individually.

In [ ]:
from phenocycler.integration.pipeline import run_pipeline

run_pipeline(cfg, only=['manifest', 'export_pheno', 'import_xenium', 'vocab',
                        'structures', 'register'], roi='panc')

## S11 — cell-to-cell pairing

Mutual-nearest centroid within a 5 µm cap, upgraded to polygon IoU where both PhenoCycler
GeoJSON and Xenium `cell_boundaries` are available.

A pair rate well below 1 is expected, not a failure: the two pipelines segment
independently — QuPath on a qptiff, Xenium's own algorithm on DAPI plus the multimodal
stains — so they will not agree on cell count.

In [ ]:
from phenocycler.integration.sameslide import run_sameslide

ss = run_sameslide(cfg, roi='panc')
ss

### The paired matrix

One row per paired cell, `protein__*` and `rna__*` side by side. This is what same-slide
mode exists to produce and what sequential mode fundamentally cannot — downstream it
supports genuinely paired modelling (totalVI-style joint latent spaces, per-cell
protein-vs-RNA discordance) that would be unjustified on serial sections.

`lineage_agreement` here is a real accuracy: two independent measurements of the *same*
cell, not a distributional similarity.

In [ ]:
import pandas as pd

path = cfg.paired_dir / 'cells.parquet'
if path.exists():
    cells = pd.read_parquet(path)
    print(f'{len(cells):,} paired cells, {cells.shape[1]} columns')
    print('protein features:', len([c for c in cells.columns if c.startswith('protein__')]))
    print('rna features    :', len([c for c in cells.columns if c.startswith('rna__')]))
    display(cells.head())

In [ ]:
from phenocycler.integration.qc import run_qc
run_qc(cfg, roi='panc')